# Aiming benchmark with a simple aimer implementation

A benchmark that tests an aiming implementation against 10_000 cases with both possible and impossible shots. It verifies the aim with the real engine and calculates accuracy.

All the board states sampled from real Kaggle replays. Maximum flight time is 20 turns. Biased towards more difficult samples with moving targets, blocking planets, commets etc.

Includes a sample fast and decent implementation of the aimer


## Setup

In [1]:
import os, sys, io, contextlib, logging
from pathlib import Path

BENCHMARK_PATH ='/kaggle/input/datasets/slawekbiel/orbit-wars-aim-benchmark/'
sys.path.insert(0, BENCHMARK_PATH)   # the standalone modules live beside this notebook

# Warm-import the kaggle engine once, silencing its env-discovery chatter
# (open_spiel logging + "Loading environment ... failed" prints).
logging.disable(logging.WARNING)
with contextlib.redirect_stdout(io.StringIO()):
    import kaggle_environments  # noqa: F401
logging.disable(logging.NOTSET)

import aim_benchmark as ab        # scores angles via the real kaggle engine
samples = list(ab.iter_samples(f'{BENCHMARK_PATH}/aim_samples.npz'))
print("benchmark samples:", len(samples))

benchmark samples: 10000


## Aimer implementation, replace with your own you want to test

It should take:
* obs - the current game state
* source - ID of the planet we aim from
* target - ID of the planet we want to hit
* fleet_size - the number of ships we want to send
  
It should return:
The angle to launch the fleet if it can reach the target, None if it's impossible



In [2]:
"""A simple, self-contained example aimer for the aim benchmark.


Strategy — iterative lead (intercept):
  1. Fleet speed is fixed by the ship count (engine formula).
  2. The target planet orbits the sun at the board centre (unless it sits at/beyond
     the rotation-radius limit, in which case it is static).
  3. Estimate the arrival time `t` from the current distance, predict where the
     target will be at `t`, recompute `t`, and repeat a few times (a contraction for
     reachable shots). Aim straight at that predicted point.

This ignores obstacles (other than the sun) and sub-pixel grazing, so it is far
from optimal — but a clean continuous lead already clears a solid fraction of the
benchmark. `aim` with your own.
"""
from __future__ import annotations

import math

# Engine constants (plain numbers — see kaggle_environments orbit_wars.py).
CENTER = 50.0
ROTATION_RADIUS_LIMIT = 50.0
DEFAULT_MAX_SHIP_SPEED = 6.0
LAUNCH_OFFSET = 0.1
SUN_RADIUS = 10.0
_LEAD_ITERS = 8


def _fleet_speed(ships: float, max_speed: float) -> float:
    """Per-turn fleet speed: 1 ship -> 1.0, scaling up to `max_speed` near 1000 ships."""
    ships = max(1.0, float(ships))
    speed = 1.0 + (max_speed - 1.0) * (math.log(ships) / math.log(1000.0)) ** 1.5
    return min(speed, max_speed)


def _seg_circle_blocked(ax, ay, bx, by, cx, cy, r) -> bool:
    """True if segment A->B passes within radius r of circle centre C."""
    dx, dy = bx - ax, by - ay
    L2 = dx * dx + dy * dy
    if L2 <= 1e-12:
        return math.hypot(cx - ax, cy - ay) < r
    t = max(0.0, min(1.0, ((cx - ax) * dx + (cy - ay) * dy) / L2))
    return math.hypot(cx - (ax + t * dx), cy - (ay + t * dy)) < r


def _planet(obs: dict, planet_id: int):
    for row in obs["planets"]:
        if int(row[0]) == int(planet_id):
            return row
    return None


def aim(obs: dict, source: int, target: int, fleet_size: int, avoid: bool = True):
    """Return a launch angle (radians) to send `fleet_size` ships from `source` at `target`.

    Default (``avoid=True``): returns ``None`` (decline) when the straight launch->lead
    segment crosses the **sun** — a rudimentary, *exact* block (the sun is static and
    central, so a sun-crossing shot always dies there; declining never costs a real
    hit). Other planets are deliberately NOT checked: they orbit, so a static "planet
    on the line" test false-declines shots that connect once the planet drifts off.
    Pass ``avoid=False`` for the bare always-shoot lead.
    """
    src = _planet(obs, source)
    tgt = _planet(obs, target)
    if src is None or tgt is None:
        return None if avoid else 0.0

    sx, sy, s_r = float(src[2]), float(src[3]), float(src[4])
    tx0, ty0, t_r = float(tgt[2]), float(tgt[3]), float(tgt[4])
    omega = float(obs.get("angular_velocity", 0.0))
    max_speed = float(obs.get("ship_speed", DEFAULT_MAX_SHIP_SPEED))
    speed = _fleet_speed(fleet_size, max_speed)

    # Target orbit: radius + phase about the centre. Static if it sits at/beyond the
    # rotation-radius limit (the engine then leaves it fixed).
    dx0, dy0 = tx0 - CENTER, ty0 - CENTER
    orbit_r = math.hypot(dx0, dy0)
    static = (orbit_r + t_r) >= ROTATION_RADIUS_LIMIT
    phase0 = math.atan2(dy0, dx0)

    def target_at(t: float):
        if static:
            return tx0, ty0
        a = phase0 + omega * t
        return CENTER + orbit_r * math.cos(a), CENTER + orbit_r * math.sin(a)

    # Distance saved by the two surfaces + the launch offset.
    gap = s_r + LAUNCH_OFFSET + t_r

    # Iterative lead: t = (distance to predicted target - gap) / speed.
    t = max(0.0, (math.hypot(tx0 - sx, ty0 - sy) - gap) / speed)
    for _ in range(_LEAD_ITERS):
        px, py = target_at(t)
        t = max(0.0, (math.hypot(px - sx, py - sy) - gap) / speed)

    px, py = target_at(t)
    angle = math.atan2(py - sy, px - sx)
    if not avoid:
        return angle

    # Rudimentary avoidance: decline if the launch->lead segment crosses the (static)
    # sun — that shot always dies there, so declining never costs a real hit.
    lx = sx + math.cos(angle) * (s_r + LAUNCH_OFFSET)
    ly = sy + math.sin(angle) * (s_r + LAUNCH_OFFSET)
    if _seg_circle_blocked(lx, ly, px, py, CENTER, CENTER, SUN_RADIUS):
        return None
    return angle

## Aim every launch with `example_aimer.aim`

In [3]:
angles = [aim(s.obs, s.source, s.target, s.fleet_size) for s in samples]
print(f"aimed {len(angles)} launches with example_aimer.aim")

aimed 10000 launches with example_aimer.aim


## Score with the benchmark (real kaggle engine)

In [4]:
results = ab.validate(angles)
correct = sum(results)
print(f"aimer accuracy: {correct}/{len(results)} = {correct/len(results):.2%}")

aimer accuracy: 7296/10000 = 72.96%
